读取 CSV，确认表头和字段类型

In [2]:
from pathlib import Path
import pandas as pd

#  Notebook 的工作目录是 q1_data_quality，向上一级找数据
if not csv_path.is_file():
    csv_path = Path("../data/online_retail_ii.csv")

if not csv_path.is_file():
    raise FileNotFoundError(f"找不到数据文件。当前工作目录为 {Path.cwd()}")

# 保留原始文本，避免把订单号、商品编码或客户 ID 自动读成数字。
# 只把空字段识别为缺失，保留 CSV 里可能存在的字面值，如 "NA"。
raw = pd.read_csv(csv_path, dtype="string", keep_default_na=False, na_values=[""])

print("数据路径", csv_path.resolve())
print("行列数", raw.shape)
print("实际表头", raw.columns.tolist())
display(raw.head())

数据路径 D:\dubf\大三\grekko lab 笔试\data-engineer-main\data\online_retail_ii.csv
行列数 (1067371, 8)
实际表头 ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


报告每列类型、缺失数和缺失率

In [3]:
df = raw.copy()

for column in ["Quantity", "Price"]:
    df[column] = pd.to_numeric(raw[column], errors="coerce")

df["InvoiceDate"] = pd.to_datetime(
    raw["InvoiceDate"], format="ISO8601", errors="coerce"
)

# 原字段有内容，但转换后变成缺失，说明该字段未能按当前规则解析。
parse_failed = raw.notna() & df.isna()

column_report = pd.DataFrame({
    "分析类型": df.dtypes.astype(str),
    "原始缺失数": raw.isna().sum(),
    "原始缺失率(%)": (raw.isna().mean() * 100).round(4),
    "解析失败数": parse_failed.sum(),
})

display(column_report)

,分析类型,原始缺失数,原始缺失率(%),解析失败数
Invoice,string,0,0.0000,0
StockCode,string,0,0.0000,0
Description,string,4382,0.4105,0
Quantity,Int64,0,0.0000,0
InvoiceDate,datetime64[ns],0,0.0000,0
Price,Float64,0,0.0000,0
Customer ID,string,243007,22.7669,0
Country,string,0,0.0000,0


主要缺失来自描述（可疑2）与用户ID（可疑3），需要进一步判断，这些内容的缺失是否会影响后续任务

探究描述缺失

删除缺描述行会让正数量、正单价口径下的金额减少多少

In [8]:
# 沿用 Q1.1 的缺失定义，和前面报告的 4,382 行保持同一口径。
missing_description = raw["Description"].isna()
positive_lines = (df["Quantity"].gt(0) & df["Price"].gt(0)).fillna(False)
affected = missing_description & positive_lines

line_amount = df["Quantity"] * df["Price"]

positive_total = line_amount.loc[positive_lines].sum()
affected_amount = line_amount.loc[affected].sum()

summary = pd.Series({
    "描述缺失行数": int(missing_description.sum()),
    "描述缺失且数量、单价均为正的行数": int(affected.sum()),
    "上述受影响行占描述缺失行的比例(%)": (
        round(100 * affected.sum() / missing_description.sum(), 4)
        if missing_description.any() else pd.NA
    ),
    "删除前正数量、正单价明细金额": round(positive_total, 2),
    "删除缺描述行将减少的金额": round(affected_amount, 2),
    "减少金额占删除前金额的比例(%)": (
        round(100 * affected_amount / positive_total, 4)
        if positive_total > 0 else pd.NA
    ),
}, name="结果")

display(summary.to_frame())

,结果
描述缺失行数,4382.00
描述缺失且数量、单价均为正的行数,0.00
上述受影响行占描述缺失行的比例(%),0.00
删除前正数量、正单价明细金额,20972968.14
删除缺描述行将减少的金额,0.00
减少金额占删除前金额的比例(%),0.00


此处ai仅给出探索正价正数量缺失描述，后续纠正探索了负数量情况

缺描述记录集中在哪些 StockCode（主要研究商品与描述之间的关系，是否存在某一类商品大量缺失描述）

In [9]:
code_summary = (
    df[["StockCode"]]
    .assign(
        描述缺失=missing_description,
        受影响正向明细=affected,
        受影响金额=line_amount.where(affected, 0),
    )
    .groupby("StockCode")
    .agg(
        该编码总行数=("描述缺失", "size"),
        描述缺失行数=("描述缺失", "sum"),
        受影响正向明细数=("受影响正向明细", "sum"),
        受影响金额=("受影响金额", "sum"),
    )
)

code_summary = code_summary.loc[code_summary["描述缺失行数"].gt(0)].copy()
code_summary["该编码描述缺失率(%)"] = (
    100 * code_summary["描述缺失行数"] / code_summary["该编码总行数"]
).round(2)

top_10 = code_summary.sort_values("描述缺失行数", ascending=False).head(10)

print("出现描述缺失的商品编码数", len(code_summary))
if missing_description.any():
    top_10_share = 100 * top_10["描述缺失行数"].sum() / missing_description.sum()
    print(f"前 10 个编码占全部描述缺失行的比例 {top_10_share:.2f}%")

display(top_10.round({"受影响金额": 2}))

出现描述缺失的商品编码数 2451
前 10 个编码占全部描述缺失行的比例 2.28%


,该编码总行数,描述缺失行数,受影响正向明细数,受影响金额,该编码描述缺失率(%)
StockCode,,,,,
84990,218,12,0,0.0,5.50
22139,2196,12,0,0.0,0.55
35965,198,11,0,0.0,5.56
79321,1222,11,0,0.0,0.90
23084,1067,10,0,0.0,0.94
22950,372,10,0,0.0,2.69
22084,852,9,0,0.0,1.06
22087,910,9,0,0.0,0.99
71477,940,8,0,0.0,0.85


有些商品虽然缺失总数较多，但是影响较小

以下对正负数量情况进行统计

In [10]:
missing_rows = df.loc[missing_description, ["Quantity", "Price"]].copy()

def sign_label(values):
    return (
        pd.Series("零", index=values.index, dtype="string")
        .mask(values.gt(0), "正")
        .mask(values.lt(0), "负")
        .mask(values.isna(), "缺失")
    )

sign_table = pd.crosstab(
    sign_label(missing_rows["Quantity"]).rename("数量符号"),
    sign_label(missing_rows["Price"]).rename("单价符号"),
    margins=True,
    margins_name="合计",
)
display(sign_table)

missing_amount = missing_rows["Quantity"] * missing_rows["Price"]
print("缺描述且明细金额非零的行数", int(missing_amount.ne(0).fillna(False).sum()))

单价符号,零,合计
数量符号,,
正,1693,1693
负,2689,2689
合计,4382,4382


缺描述且明细金额非零的行数 0


所有缺描述行的单价均为 0（可疑4）；删行对明细金额合计无影响，但会丢失正负数量记录

Description 缺失的 4,382 条明细，占全表的 0.4105%，涉及 2,451 个 StockCode。将数量和单价按符号交叉统计后发现，这些明细的单价全部为 0，其中 1,693 条数量为正，2,689 条为负。各行按 Quantity × Price 计算的金额均为 0，但直接删除会丢失这些数量变动记录。仅凭当前字段，无法确认这些变动属于退货、库存调整，还是其他业务操作。

用户ID缺失影响（可疑3）

In [11]:
missing_customer = raw["Customer ID"].isna()
is_cancellation = df["Invoice"].str.strip().str.upper().str.startswith("C", na=False)
line_amount = df["Quantity"] * df["Price"]

scopes = {
    "全量明细金额（含取消及负值）": pd.Series(True, index=df.index),
    "正向明细金额（排除取消、数量和单价均为正）": (
        ~is_cancellation & df["Quantity"].gt(0) & df["Price"].gt(0)
    ).fillna(False),
}

results = []

for scope_name, scope_mask in scopes.items():
    total_rows = int(scope_mask.sum())
    total_amount = line_amount.loc[scope_mask].sum()

    for group_name, customer_mask in [
        ("客户 ID 缺失", missing_customer),
        ("客户 ID 非缺失", ~missing_customer),
    ]:
        group_mask = scope_mask & customer_mask
        group_rows = int(group_mask.sum())
        group_amount = line_amount.loc[group_mask].sum()

        results.append({
            "统计口径": scope_name,
            "客户组": group_name,
            "明细行数": group_rows,
            "行数占比(%)": 100 * group_rows / total_rows if total_rows else pd.NA,
            "明细金额": group_amount,
            "金额占比(%)": 100 * group_amount / total_amount
            if total_amount != 0 else pd.NA,
        })

coverage_report = pd.DataFrame(results)
display(coverage_report.round({
    "行数占比(%)": 4,
    "明细金额": 2,
    "金额占比(%)": 4,
}))

,统计口径,客户组,明细行数,行数占比(%),明细金额,金额占比(%)
0,全量明细金额（含取消及负值）,客户 ID 缺失,243007,22.7669,2638958.18,13.6824
1,全量明细金额（含取消及负值）,客户 ID 非缺失,824364,77.2331,16648292.39,86.3176
2,正向明细金额（排除取消、数量和单价均为正）,客户 ID 缺失,236121,22.6675,3229165.39,15.3971
3,正向明细金额（排除取消、数量和单价均为正）,客户 ID 非缺失,805549,77.3325,17743429.18,84.6029


正向口径下，缺失组的金额占比比行数占比低 7.2704 个百分点,这说明缺失组的每条明细平均金额低于非缺失组

ID缺失占比过大不能直接去除，需要根据数据用途决定数据处理方向，留在后续考虑

补：考虑到q1.3存在下游用途分类，此处数据处理需要考虑三种不同情况

注：此表由ai生成，后续需进一步核实
| 下游用途 | 对 `Customer ID` 的要求 | 缺失时怎么处理 |
|---|---|---|
| **A. 销售总额统计 / 报表** | **汇总销售总额时不必有客户 ID。** 金额按符合报表口径的交易明细汇总。如果报表还要统计独立客户数、复购率，就需要客户 ID。 | 保留符合销售口径的缺 ID 明细。可以单独报告这部分金额，让读者知道有多少销售无法归属到具体客户。 |
| **B. 客户分群（RFM）** | **需要稳定、可跨订单关联的客户 ID。** RFM 要知道同一客户最近何时购买、买过几次、累计消费多少。 | 缺 ID 的明细无法归到某个客户，因此不计入个人 RFM。不要把所有缺失值填成同一个 `UNKNOWN` 后参与分群，否则会把不同人的交易合成一个虚构客户。 |
| **C. 商品推荐模型** | **取决于推荐方法。** 基于用户历史的个性化推荐需要稳定的客户 ID。购物篮共现推荐只需知道哪些商品出现在同一张有效发票上，通常用 `Invoice` 分组。 | 缺 ID 的交易不能用于构造个人购买历史，但仍可按有效发票贡献商品共现信号，也可以用于统计商品热度。 |

统计数值分布和字符串 TOP 10

In [4]:
numeric_columns = ["Quantity", "Price"]
numeric_report = df[numeric_columns].agg(["min", "max", "mean", "std"]).T
numeric_report.columns = ["最小值", "最大值", "均值", "样本标准差"]
display(numeric_report)

text_columns = df.select_dtypes(include="string").columns

text_summary = pd.DataFrame({
    "非空唯一值数": df[text_columns].nunique(dropna=True)
})
display(text_summary)

for column in text_columns:
    print(column)
    top_10 = (
        df[column]
        .value_counts(dropna=True)
        .head(10)
        .rename_axis("原始值")
        .reset_index(name="出现行数")
    )
    display(top_10)

,最小值,最大值,均值,样本标准差
Quantity,-80995.00,80995.0,9.938898,172.705794
Price,-53594.36,38970.0,4.649388,123.553059


,非空唯一值数
Invoice,53628
StockCode,5305
Description,5698
Customer ID,5942
Country,43


Invoice


,原始值,出现行数
0,537434,1350
1,538071,1304
2,537638,1202
3,537237,1194
4,536876,1186
5,536592,1184
6,537823,1182
7,537240,1136
8,573585,1114
9,537666,1072


StockCode


,原始值,出现行数
0,85123A,5829
1,22423,4424
2,85099B,4216
3,21212,3318
4,20725,3259
5,84879,2960
6,47566,2768
7,21232,2747
8,22197,2549
9,22383,2540


... (2 outputs left out)

Customer ID


,原始值,出现行数
0,17841.0,13097
1,14911.0,11613
2,12748.0,7307
3,14606.0,6709
4,14096.0,5128
5,15311.0,4717
6,14156.0,4130
7,14646.0,3890
8,13089.0,3438
9,16549.0,3255


Country


,原始值,出现行数
0,United Kingdom,981330
1,EIRE,17866
2,Germany,17624
3,France,14330
4,Netherlands,5140
5,Spain,3811
6,Switzerland,3189
7,Belgium,3123
8,Portugal,2620
9,Australia,1913


对上一单元格进行进一步探索--为什么最大数量与最小数量绝对值相同，是否来自同一笔订单？（可疑1）

In [7]:
columns = [
    "Invoice", "StockCode", "Customer ID",
    "Description", "Quantity", "Price", "InvoiceDate", "Country",
]

max_qty = df["Quantity"].max()
min_qty = df["Quantity"].min()

positive = df.loc[df["Quantity"].eq(max_qty), columns].copy()
negative = df.loc[df["Quantity"].eq(min_qty), columns].copy()

print(f"最大数量 {max_qty}，命中 {len(positive)} 行")
display(positive)

print(f"最小数量 {min_qty}，命中 {len(negative)} 行")
display(negative)

# 检查两端极值是否共享同一个发票号。
shared_invoices = set(positive["Invoice"].dropna()) & set(negative["Invoice"].dropna())
print("两端极值共有的发票号", sorted(shared_invoices))

# 用非空客户 ID 和商品编码建立候选配对。
# 先排除缺失值，避免 pandas 将两边的空键匹配起来。
keys = ["Customer ID", "StockCode"]
pairs = positive.dropna(subset=keys).merge(
    negative.dropna(subset=keys),
    on=keys,
    how="inner",
    suffixes=("_正向", "_负向"),
)

pairs["数量正好抵消"] = pairs["Quantity_正向"].eq(-pairs["Quantity_负向"])
pairs["单价相同"] = pairs["Price_正向"].eq(pairs["Price_负向"])
pairs["时间间隔"] = pairs["InvoiceDate_负向"] - pairs["InvoiceDate_正向"]
pairs["负向发生在正向之后"] = pairs["时间间隔"].gt(pd.Timedelta(0))
pairs["负向发票以C开头"] = pairs["Invoice_负向"].str.startswith("C", na=False)

print("同客户、同商品的极值候选配对数", len(pairs))
display(pairs[[
    "Invoice_正向", "Invoice_负向", "Customer ID", "StockCode",
    "Quantity_正向", "Quantity_负向", "Price_正向", "Price_负向",
    "InvoiceDate_正向", "InvoiceDate_负向", "时间间隔",
    "数量正好抵消", "单价相同", "负向发生在正向之后", "负向发票以C开头",
]])

最大数量 80995，命中 1 行


,Invoice,StockCode,Customer ID,Description,Quantity,Price,InvoiceDate,Country
1065882,581483,23843,16446.0,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00,United Kingdom


最小数量 -80995，命中 1 行


,Invoice,StockCode,Customer ID,Description,Quantity,Price,InvoiceDate,Country
1065883,C581484,23843,16446.0,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00,United Kingdom


两端极值共有的发票号 []
同客户、同商品的极值候选配对数 1


,Invoice_正向,Invoice_负向,Customer ID,StockCode,Quantity_正向,Quantity_负向,Price_正向,Price_负向,InvoiceDate_正向,InvoiceDate_负向,时间间隔,数量正好抵消,单价相同,负向发生在正向之后,负向发票以C开头
0,581483,C581484,16446.0,23843,80995,-80995,2.08,2.08,2011-12-09 09:15:00,2011-12-09 09:27:00,0 days 00:12:00,True,True,True,True


运行证明是同一笔订单的下单与取消，虽然数量极大，但是不是异常值

In [5]:
checks = {
    "数量等于最小值": df["Quantity"].eq(df["Quantity"].min()),
    "数量等于最大值": df["Quantity"].eq(df["Quantity"].max()),
    "单价为负": df["Price"].lt(0),
    "数量为正但单价为零": df["Quantity"].gt(0) & df["Price"].eq(0),
    "整行内容重复": raw.duplicated(keep=False),
    "商品描述缺失但数量和单价均为正": (
        raw["Description"].isna() & df["Quantity"].gt(0) & df["Price"].gt(0)
    ),
    "有内容但只包含空白字符的描述": raw["Description"].str.fullmatch(r"\s+", na=False),
}

evidence_columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country",
]

for name, mask in checks.items():
    mask = mask.fillna(False)
    print(name, "命中行数", int(mask.sum()))
    if mask.any():
        display(raw.loc[mask, evidence_columns].head(5))

数量等于最小值 命中行数 1


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


数量等于最大值 命中行数 1


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom


单价为负 命中行数 5


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,<NA>,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,<NA>,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,<NA>,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom


数量为正但单价为零 命中行数 2745


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
3161,489659,21350,<NA>,230,2009-12-01 17:39:00,0.0,<NA>,United Kingdom
3731,489781,84292,<NA>,17,2009-12-02 11:45:00,0.0,<NA>,United Kingdom
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,<NA>,United Kingdom
6378,489882,35751C,<NA>,12,2009-12-02 16:22:00,0.0,<NA>,United Kingdom


整行内容重复 命中行数 67242


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


商品描述缺失但数量和单价均为正 命中行数 0
有内容但只包含空白字符的描述 命中行数 0


观察第三个表很明显存在一个现象，单价为负的时候，Invoice以A开头，StockCode为B，Description为Adjust bad debt，Quantity为1（可疑6）

考虑是否存在‘错误’数据（即1对多的情况）

In [12]:
descriptions = raw[["StockCode", "Description"]].copy()

# 保留原始描述。另建一列，仅用于比较大小写和空白差异。
descriptions["规范化描述"] = (
    descriptions["Description"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.casefold()
    .replace("", pd.NA)
)

code_report = descriptions.groupby("StockCode").agg(
    原始描述数=("Description", "nunique"),
    规范化描述数=("规范化描述", "nunique"),
)

raw_multiple = code_report["原始描述数"].gt(1)
normalized_multiple = code_report["规范化描述数"].gt(1)

format_resolved = raw_multiple & code_report["规范化描述数"].eq(1)

print("参与统计的编码数", len(code_report))
print("对应多个原始非缺失描述的编码数", int(raw_multiple.sum()))
print("规范化后只剩一个描述的多描述编码数", int(format_resolved.sum()))
print("规范化后仍对应多个描述的编码数", int(normalized_multiple.sum()))

display(
    code_report.loc[raw_multiple]
    .sort_values(["规范化描述数", "原始描述数"], ascending=False)
    .head(10)
)

参与统计的编码数 5305
对应多个原始非缺失描述的编码数 1232
规范化后只剩一个描述的多描述编码数 22
规范化后仍对应多个描述的编码数 1210


,原始描述数,规范化描述数
StockCode,,
20713,9,8
22423,7,7
22734,7,7
23084,7,7
21181,7,6
21830,6,6
22719,6,6
47566B,6,6
85175,6,6


挑一个规范化后仍有多个描述的编码回查

In [13]:
remaining = code_report.loc[normalized_multiple]

if remaining.empty:
    print("规范化后没有发现对应多个描述的编码")
else:
    # 默认选规范化后描述最多的编码，也可以手动换成你感兴趣的编码。
    example_code = remaining["规范化描述数"].idxmax()
    print("示例编码", example_code)

    example_rows = descriptions.loc[
        descriptions["StockCode"].eq(example_code)
        & descriptions["Description"].notna()
    ].copy()

    # repr 会保留引号，并让前后空格、制表符等更容易辨认。
    example_rows["原始描述（显示引号）"] = example_rows["Description"].map(repr)

    variants = (
        example_rows
        .groupby(["原始描述（显示引号）", "规范化描述"], dropna=False)
        .size()
        .rename("出现行数")
        .reset_index()
        .sort_values("出现行数", ascending=False)
    )

    display(variants.head(15))

示例编码 20713


,原始描述（显示引号）,规范化描述,出现行数
1,'JUMBO BAG OWLS',jumbo bag owls,1372
0,'Found',found,1
2,'Marked as 23343',marked as 23343,1
3,'found',found,1
4,'missing',missing,1
5,'wrongly coded 23343',wrongly coded 23343,1
6,'wrongly coded-23343',wrongly coded-23343,1
7,'wrongly marked 23343',wrongly marked 23343,1
8,'wrongly marked. 23343 in box',wrongly marked. 23343 in box,1


此处发现规范后任存在相同含义描述‘wrongly coded 23343’‘wrongly coded-23343’‘wrongly marked 23343’实际上是一个东西（可疑5）